# 01 — Quality Gate 1: ворота якості, евристичний бейзлайн, аудиторський слід

Демонстрація вимог п. 3.4.9 ТЗ: спрацювання `validate_config` на зіпсованій конфігурації, збирання без звернення до даних, навчання адаптивного імпутера лише на train, аудиторський слід, перевірка схеми ключів `scale_sensitive / scale_invariant` тестом на множення колонки на константу.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore", message="Bins whose width are too small")
pd.set_option("display.width", 160)
df = pd.read_csv(ROOT / "data/processed/ecommerce_churn_mar.csv")
manifest = json.loads((ROOT / "data/processed/injection_manifest.json").read_text(encoding="utf-8"))
df.shape, manifest["churn_share"]

((5630, 19), 0.1684)

## 1. Евристичний бейзлайн (без навчання, на всьому наборі)

Правило: клієнт у зоні відтоку, якщо він з нами ≤ 1 місяць **або** подавав скаргу і при цьому був активний в останні 3 дні (скарга після нещодавнього замовлення). Метрики рахуються для класу `Churn = 1`.

In [2]:
from sklearn.metrics import accuracy_score, recall_score, precision_score
y = df['Churn']
rule = (df['Tenure'].le(1) | (df['Complain'].eq(1) & df['DaySinceLastOrder'].le(3))).fillna(False).astype(int)
print('Accuracy:', round(accuracy_score(y, rule), 3))
print('Recall(churn):', round(recall_score(y, rule), 3))
print('Precision(churn):', round(precision_score(y, rule), 3))
print('частка позначених:', round(rule.mean(), 3), '| тривіальне «ніхто не піде»: Accuracy', round(1 - y.mean(), 3))

Accuracy: 0.781
Recall(churn): 0.778
Precision(churn): 0.42
частка позначених: 0.312 | тривіальне «ніхто не піде»: Accuracy 0.832


## 2. Спрацювання воріт якості

У `tests/fixtures/config_protected_leak.yaml` захищена ознака `CityTier` внесена до `features`. Конструктор має зупинитись до збирання конвеєра.

In [3]:
from src.pipeline_builder import DynamicPipelineBuilder, ConfigError, audit_trail
try:
    DynamicPipelineBuilder(str(ROOT / 'tests/fixtures/config_protected_leak.yaml'))
except ValueError as e:
    print('ValueError:', e)

ValueError: Захищені ознаки потрапили у features: {'CityTier'}


Друга перевірка — структурна: невідомий крок у `transform_policy` або неупорядкована політика імпутації також зупиняють конструктор (`ConfigError`), тож помилка конфігурації не доживає до навчання.

In [4]:
import yaml, tempfile
cfg = yaml.safe_load(open(ROOT / 'config/config_lr.yaml', encoding='utf-8'))
cfg['transform_policy']['scale_sensitive']['numeric_discrete'] = ['magic_scaler']
with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False, encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)
try:
    DynamicPipelineBuilder(f.name)
except ConfigError as e:
    print('ConfigError:', e)

ConfigError: невідомий крок 'magic_scaler' у групі 'numeric_discrete'


## 3. Збирання без даних, навчання лише на train

`build()` повертає ненавчений `ColumnTransformer`; розбиття 70/30 (stratify, seed 42) — перша випадковість у курсі з'являється саме тут, але лише для демонстрації: класифікатори не навчаються.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError
X = df.drop(columns=['Churn']); y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
ct = DynamicPipelineBuilder(str(ROOT / 'config/config_lr.yaml')).build()
try:
    check_is_fitted(ct); print('навчений?!')
except NotFittedError:
    print('build() повернув ненавчений ColumnTransformer')
Z_train = ct.fit_transform(X_train, y_train); Z_test = ct.transform(X_test)
print('train', Z_train.shape, 'test', Z_test.shape, '| NaN у виході:', int(np.isnan(np.asarray(Z_train, float)).sum()))

build() повернув ненавчений ColumnTransformer
train (3941, 27) test (1689, 27) | NaN у виході: 0


## 4. Аудиторський слід (п. 3.4.9, крок 4)

Частки пропусків обчислені імпутером **на тренувальній підвибірці** і відрізняються від часток у повному наборі. У гілці `scale_sensitive` політика диференційована: індикатор створюється навіть при малій частці пропусків, бо медіанна заміна зміщує розподіл для агрегуючої моделі.

In [6]:
tr = audit_trail(ct)
tr

,group,missing_share,rule_index,max_share,strategy,fill_value,indicator
column,,,,,,,
Tenure,numeric_continuous,0.0462,0,0.1,median,NaN,True
WarehouseToHome,numeric_continuous,0.0472,0,0.1,median,NaN,True
CashbackAmount,numeric_continuous,0.2172,1,0.4,median,NaN,True
OrderAmountHikeFromlastYear,numeric_continuous,0.6308,2,1.0,constant,0.0,True
DaySinceLastOrder,numeric_continuous,0.0543,0,0.1,median,NaN,True
HourSpendOnApp,numeric_discrete,0.0439,0,0.1,median,None,True
NumberOfDeviceRegistered,numeric_discrete,0.0000,0,0.1,median,None,True
NumberOfAddress,numeric_discrete,0.0000,0,0.1,median,None,True
CouponUsed,numeric_discrete,0.0464,0,0.1,median,None,True


In [7]:
ct_inv = DynamicPipelineBuilder(str(ROOT / 'config/config_rf.yaml')).build().fit(X_train, y_train)
tr_inv = audit_trail(ct_inv)
cmp = tr[['group','missing_share','rule_index','strategy','indicator']].rename(columns={'indicator':'indicator_sensitive'})
cmp['indicator_invariant'] = tr_inv['indicator']
cmp['rules_shown'] = cmp['rule_index'].map({0:'≤10%', 1:'10–40%', 2:'>40%'})
cmp

,group,missing_share,rule_index,strategy,indicator_sensitive,indicator_invariant,rules_shown
column,,,,,,,
Tenure,numeric_continuous,0.0462,0,median,True,False,≤10%
WarehouseToHome,numeric_continuous,0.0472,0,median,True,False,≤10%
CashbackAmount,numeric_continuous,0.2172,1,median,True,True,10–40%
OrderAmountHikeFromlastYear,numeric_continuous,0.6308,2,constant,True,True,>40%
DaySinceLastOrder,numeric_continuous,0.0543,0,median,True,False,≤10%
HourSpendOnApp,numeric_discrete,0.0439,0,median,True,False,≤10%
NumberOfDeviceRegistered,numeric_discrete,0.0000,0,median,True,False,≤10%
NumberOfAddress,numeric_discrete,0.0000,0,median,True,False,≤10%
CouponUsed,numeric_discrete,0.0464,0,median,True,False,≤10%


Продемонстровані гілки: числові — всі три (`≤ 10 %`: Tenure, WarehouseToHome, …; `10–40 %`: CashbackAmount; `> 40 %`: OrderAmountHikeFromlastYear), категоріальні — обидві (`≤ 10 %`: PreferredLoginDevice, PreferedOrderCat, MaritalStatus; `> 10 %`: PreferredPaymentMode → категорія Missing).

## 5. Перевірка схеми ключів: тест на множення колонки на константу

Критерій ТЗ (п. 3.4.4): якщо помножити одну числову ознаку на константу, вихід `scale_invariant`-гілки не має змінитись, а `scale_sensitive` — має.

In [8]:
Xs_train, Xs_test = X_train.copy(), X_test.copy()
Xs_train['CashbackAmount'] *= 1000; Xs_test['CashbackAmount'] *= 1000
for name in ['config_lr', 'config_rf']:
    path = str(ROOT / f'config/{name}.yaml')
    a = DynamicPipelineBuilder(path).build().fit(X_train, y_train).transform(X_test)
    b = DynamicPipelineBuilder(path).build().fit(Xs_train, y_train).transform(Xs_test)
    fam = yaml.safe_load(open(path, encoding='utf-8'))['model_family']
    print(f'{fam:16s} max|Δ| = {np.abs(np.asarray(a, float) - np.asarray(b, float)).max():.3g}')

scale_sensitive  max|Δ| = 0.315
scale_invariant  max|Δ| = 0


## 6. Вихідна схема конвеєра

Жодної захищеної ознаки серед вихідних імен; кількість ознак відрізняється між родинами через індикатори пропусків і спосіб кодування.

In [9]:
names = list(ct.get_feature_names_out()); names_inv = list(ct_inv.get_feature_names_out())
print(len(names), 'ознак (scale_sensitive):', names)
print(len(names_inv), 'ознак (scale_invariant):', names_inv)
assert not any(p in n for n in names + names_inv for p in ['CityTier', 'Gender'])

27 ознак (scale_sensitive): ['numeric_continuous__Tenure', 'numeric_continuous__WarehouseToHome', 'numeric_continuous__DaySinceLastOrder', 'numeric_continuous__Tenure__missing', 'numeric_continuous__WarehouseToHome__missing', 'numeric_continuous__DaySinceLastOrder__missing', 'numeric_continuous__CashbackAmount', 'numeric_continuous__CashbackAmount__missing', 'numeric_continuous__OrderAmountHikeFromlastYear', 'numeric_continuous__OrderAmountHikeFromlastYear__missing', 'numeric_discrete__HourSpendOnApp', 'numeric_discrete__NumberOfDeviceRegistered', 'numeric_discrete__NumberOfAddress', 'numeric_discrete__CouponUsed', 'numeric_discrete__OrderCount', 'numeric_discrete__HourSpendOnApp__missing', 'numeric_discrete__NumberOfDeviceRegistered__missing', 'numeric_discrete__NumberOfAddress__missing', 'numeric_discrete__CouponUsed__missing', 'numeric_discrete__OrderCount__missing', 'binary_flag__Complain', 'binary_flag__Complain__missing', 'categorical_nominal__PreferredLoginDevice_freq', 'categor